In [0]:
# ============================================================
# Cell 1 - Configuration
# ============================================================
from pyspark.sql.functions import from_json, explode, col, to_date, when
from pyspark.sql.types import (StructType, StructField, StringType,
                                IntegerType, ArrayType, DoubleType, BooleanType)

CATALOG = "media"
BRONZE  = "bronze_tmdb"
SILVER  = "silver_tmdb"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

print(f"Ready: {CATALOG}.{SILVER}")

In [0]:
# ============================================================
# Cell 2 - Silver movies
# ============================================================
json_schema_movies = StructType([
    StructField("adult",             BooleanType(),            True),
    StructField("backdrop_path",     StringType(),             True),
    StructField("genre_ids",         ArrayType(IntegerType()), True),
    StructField("id",                IntegerType(),            True),
    StructField("original_language", StringType(),             True),
    StructField("original_title",    StringType(),             True),
    StructField("overview",          StringType(),             True),
    StructField("popularity",        DoubleType(),             True),
    StructField("poster_path",       StringType(),             True),
    StructField("release_date",      StringType(),             True),
    StructField("title",             StringType(),             True),
    StructField("video",             BooleanType(),            True),
    StructField("vote_average",      DoubleType(),             True),
    StructField("vote_count",        IntegerType(),            True),
])

raw_movies = spark.table(f"{CATALOG}.{BRONZE}.raw_movies")
parsed     = raw_movies.withColumn("p", from_json(col("raw_payload"), json_schema_movies))

silver_movies = parsed.select(
    col("p.id").alias("movie_id"),
    col("p.title"),
    col("p.original_title"),
    col("p.overview"),
    to_date(col("p.release_date"), "yyyy-MM-dd").alias("release_date"),
    col("p.original_language"),
    col("p.popularity"),
    col("p.vote_average"),
    col("p.vote_count"),
    col("p.adult"),
    col("p.video"),
    col("p.backdrop_path"),
    col("p.poster_path"),
    col("p.genre_ids"),
    col("ingested_at")
).dropDuplicates(["movie_id"])

(silver_movies.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.silver_movies"))

print(f"silver_movies: {silver_movies.count():,} rows")

In [0]:
# ============================================================
# Cell 3 - Silver movie genres bridge table
# ============================================================
genres_movie = spark.table(f"{CATALOG}.{BRONZE}.raw_genres_movie")

silver_movie_genres = (parsed
    .select(col("p.id").alias("movie_id"), explode(col("p.genre_ids")).alias("genre_id"))
    .join(genres_movie, col("genre_id") == genres_movie.id)
    .select("movie_id", "genre_id", col("name").alias("genre_name"))
    .dropDuplicates(["movie_id", "genre_id"]))

(silver_movie_genres.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.silver_movie_genres"))

print(f"silver_movie_genres: {silver_movie_genres.count():,} rows")

In [0]:
# ============================================================
# Cell 4 - Silver TV shows
# ============================================================
json_schema_tv = StructType([
    StructField("adult",             BooleanType(),            True),
    StructField("backdrop_path",     StringType(),             True),
    StructField("genre_ids",         ArrayType(IntegerType()), True),
    StructField("id",                IntegerType(),            True),
    StructField("origin_country",    ArrayType(StringType()),  True),
    StructField("original_language", StringType(),             True),
    StructField("original_name",     StringType(),             True),
    StructField("overview",          StringType(),             True),
    StructField("popularity",        DoubleType(),             True),
    StructField("poster_path",       StringType(),             True),
    StructField("first_air_date",    StringType(),             True),
    StructField("name",              StringType(),             True),
    StructField("vote_average",      DoubleType(),             True),
    StructField("vote_count",        IntegerType(),            True),
])

raw_tv = spark.table(f"{CATALOG}.{BRONZE}.raw_tv_shows")
parsed_tv = raw_tv.withColumn("p", from_json(col("raw_payload"), json_schema_tv))

silver_tv_shows = parsed_tv.select(
    col("p.id").alias("show_id"),
    col("p.name").alias("title"),
    col("p.original_name").alias("original_title"),
    col("p.overview"),
    to_date(
        when(col("p.first_air_date") != "", col("p.first_air_date")),
        "yyyy-MM-dd"
    ).alias("first_air_date"),
    col("p.original_language"),
    col("p.origin_country"),
    col("p.popularity"),
    col("p.vote_average"),
    col("p.vote_count"),
    col("p.adult"),
    col("p.backdrop_path"),
    col("p.poster_path"),
    col("p.genre_ids"),
    col("ingested_at")
).dropDuplicates(["show_id"])

(silver_tv_shows.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.silver_tv_shows"))

print(f"silver_tv_shows: {silver_tv_shows.count():,} rows")

In [0]:
# ============================================================
# Cell 5 - Silver TV genres bridge table
# ============================================================
genres_tv = spark.table(f"{CATALOG}.{BRONZE}.raw_genres_tv")

silver_tv_genres = (parsed_tv
    .select(col("p.id").alias("show_id"), explode(col("p.genre_ids")).alias("genre_id"))
    .join(genres_tv, col("genre_id") == genres_tv.id)
    .select("show_id", "genre_id", col("name").alias("genre_name"))
    .dropDuplicates(["show_id", "genre_id"]))

(silver_tv_genres.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER}.silver_tv_genres"))

print(f"silver_tv_genres: {silver_tv_genres.count():,} rows")

In [0]:
# ============================================================
# Cell 6 - Sanity check
# ============================================================
print("=== Silver Layer Summary ===")
for table in ["silver_movies", "silver_movie_genres", "silver_tv_shows", "silver_tv_genres"]:
    count = spark.table(f"{CATALOG}.{SILVER}.{table}").count()
    print(f"  {CATALOG}.{SILVER}.{table}: {count:,} rows")